In [0]:
%pip install polars deltalake tpot scikit-learn

In [0]:
dbutils.library.restartPython()

In [ ]:
experiment_size = 25 # choose from 25, 50, 200, 250

In [0]:
import sys
sys.path.insert(0,'/home/jupyter/repos/synthetic-data-generator')
data_volume = f"/Volumes/EntityML/entity_scale_icde2025/{experiment_size}"

In [0]:
# ray.util.spark.shutdown_ray_cluster()

In [0]:
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

setup_ray_cluster(
  max_worker_nodes=5,
  min_worker_nodes=5,
  num_cpus_worker_node=16,
  num_gpus_worker_node=0,
  collect_log_to_path="/Volumes/EntityML/entity_scale_icde2025/logs"
)

# Pass any custom Ray configuration with ray.init
ray.init(ignore_reinit_error=True)

In [0]:
import datetime
from pathlib import Path
import pandas as pd
import polars as pl
from deltalake import DeltaTable
from tpot import TPOTClassifier, TPOTRegressor
from sklearn.model_selection import train_test_split


In [0]:
input_url = data_volume + "/regression"

dt = DeltaTable(input_url)
# dict of partitions
entity_partitions = dt.partitions()
sample_fraction = 1.0
entity_col = 'entity'

tpot = TPOTRegressor(
    generations=5, population_size=50, verbosity=2, random_state=42, n_jobs=4, config_dict='TPOT light',
)

In [0]:
@ray.remote(num_cpus=4)
def process_single_entity(input_url, entity_col, entity_partition, sample_fraction):
  entity_start_time =  datetime.datetime.now()
  entity_pldf = pl.read_delta(input_url, use_pyarrow=True, pyarrow_options={"partitions": [(f"{entity_col}", "=", f"{entity_partition}")]})
  entity_pdf = entity_pldf.sample(fraction=sample_fraction).to_pandas()
  
  features = [s for s in entity_pdf.columns.to_list() if s.isdigit()]
  target = 'target'

  # prepare data
  X_train, X_test, y_train, y_test = train_test_split(
      entity_pdf[features], entity_pdf[target], test_size=0.25, random_state=42
  )

  tpot.fit(X_train, y_train)
  entity_score = tpot.score(X_test, y_test)
  entity_total_time =  datetime.datetime.now() - entity_start_time
  return entity_partition, entity_pldf.shape, entity_pdf.shape, entity_score, entity_total_time

In [0]:
output = {}
output_list = []
start_time_ray =  datetime.datetime.now()

for part in entity_partitions:
    entity_partition = part[entity_col]
    entity_objref = process_single_entity.remote(input_url, entity_col, entity_partition, sample_fraction)
    output[entity_objref] = (input_url, entity_partition, sample_fraction)
    output_list.append(entity_objref)

In [0]:
results = {}
while output:
    done, _ = ray.wait(list(output.keys()), num_returns=1)
    done = done[0]
    done_key = output.pop(done)
    print(f"original args ..... {done_key}")
    try:
        results[done_key[1]] = ray.get(done)
    except (ray.exceptions.RayTaskError, ray.exceptions.WorkerCrashedError) as e:
        # get raised memory error and then retry based on that
        print(e)
        input_data_path, entity_col, entity_partition, sample_fraction = done_key
        sample_fraction *= 0.9
        done_key = (input_data_path, entity_col, entity_partition, sample_fraction)
        entity_objref = process_single_entity.remote(*done_key)
        output[entity_objref] = done_key
        print(f"updated args ------ {done_key}")

exec_duration_ray =  datetime.datetime.now() - start_time_ray

In [0]:
exec_duration_ray

In [0]:
results

In [0]:
#1 datetime.timedelta(seconds=15895, microseconds=561225)
#2 datetime.timedelta(seconds=15622, microseconds=980892)
#3 datetime.timedelta(seconds=15770, microseconds=421372)